# Challenge Task — Section 3: Post-Processing (Median Filter)

The per-segment HistGB predictions are produced independently for each
1-second window — they are noisy and temporally inconsistent. We apply a
**median filter** along the time axis (per class), which replaces each
prediction by the median of its neighbours. With an odd window of size
`w`, a 1 is kept only if more than `w/2` neighbouring segments are also
1, so isolated false positives are removed while events of length ≥`w/2`
survive.

We:
1. Reload the best HistGB model + per-class thresholds from Section 2.
2. Sweep `median_window ∈ {1, 3, 5, 7, 9}` on the local val set.
3. Report the macro-F1 and identify which classes benefit most.
4. Lock the best window for the final hidden-test submission.

## 1 — Setup, reload best model + thresholds

In [ ]:
import os, sys, time, glob, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

PHASE = Path('/Users/amrabdullah/AI_2026ss/mlpc/phases/04_challenge')
sys.path.insert(0, str(PHASE))
import utils

ART = PHASE / 'artifacts'
hgb  = joblib.load(ART / 'section2_histgb_best.joblib')
thr  = np.load(ART / 'section2_thresholds.npy')
print('best HistGB + thresholds loaded; thr min/max:', round(thr.min(), 3), round(thr.max(), 3))

our_val_files, our_nht_files = utils.split_validation(seed=42)
ann_df_val = pd.read_csv(utils.PATH_VAL / 'annotations.csv')
print(f'val files: {len(our_val_files)}, non-hidden test files: {len(our_nht_files)}')

## 2 — Median-window sweep on local validation

In [ ]:
WINDOWS = [1, 3, 5, 7, 9]
sweep = []
per_class_collect = {}

for w in WINDOWS:
    t0 = time.time()
    pred = utils.generate_predictions(our_val_files, hgb, thresholds=thr,
                                       median_window=w, use_probs=True)
    f1, per = utils.evaluate_split(pred, our_val_files, ann_df_val)
    sweep.append({'window': w, 'val_macro_f1': float(f1),
                  'n_intervals': int(len(pred)),
                  'seconds': round(time.time() - t0, 1)})
    per_class_collect[w] = per[['annotation', 'f1']].set_index('annotation')['f1']
    print(f'  window={w}  ->  val macro-F1 = {f1:.4f}  ({len(pred)} intervals)  '
          f'({time.time()-t0:.0f}s)')

sweep_df = pd.DataFrame(sweep)
sweep_df.to_csv(ART / 'section3_window_sweep.csv', index=False)
sweep_df

In [ ]:
# Plot the sweep
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(sweep_df['window'], sweep_df['val_macro_f1'], marker='o', lw=2)
ax.set_xticks(WINDOWS)
ax.set_xlabel('Median filter window (segments, 1 s each)')
ax.set_ylabel('Macro-F1 (local validation)')
ax.set_title('Median-filter window sweep — local validation')
ax.grid(alpha=0.3)
for _, row in sweep_df.iterrows():
    ax.annotate(f"{row['val_macro_f1']:.3f}",
                (row['window'], row['val_macro_f1']),
                textcoords='offset points', xytext=(0, 8), ha='center')
plt.tight_layout()
fig.savefig(PHASE / 'report' / 'figures' / 'section3_window_sweep.png',
            dpi=150, bbox_inches='tight')
plt.show()

## 3 — Per-class deltas (which classes benefit from filtering)

In [ ]:
# Compare window=1 (no filter) vs window=best on a per-class basis
best_w = int(sweep_df.sort_values('val_macro_f1', ascending=False).iloc[0]['window'])
print(f'Best window: {best_w}')

baseline_per = per_class_collect[1]
best_per     = per_class_collect[best_w]
delta = (best_per - baseline_per).rename(f'delta_w{best_w}_vs_w1')

cmp = pd.DataFrame({'class': utils.CLASS_NAMES,
                    'f1_w1':     baseline_per.reindex(utils.CLASS_NAMES).values,
                    f'f1_w{best_w}': best_per.reindex(utils.CLASS_NAMES).values})
cmp[f'delta'] = cmp[f'f1_w{best_w}'] - cmp['f1_w1']
cmp.sort_values('delta', ascending=False, inplace=True)
print(cmp.to_string(index=False))
cmp.to_csv(ART / 'section3_per_class_delta.csv', index=False)

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.bar(np.arange(len(cmp)), cmp['delta'].values, color='steelblue')
ax.axhline(0, color='black', lw=0.8)
ax.set_xticks(np.arange(len(cmp)))
ax.set_xticklabels(cmp['class'].values, rotation=45, ha='right')
ax.set_ylabel(f'F1 delta (w={best_w} - w=1)')
ax.set_title(f'Per-class effect of median filter (window {best_w}) on local validation')
plt.tight_layout()
fig.savefig(PHASE / 'report' / 'figures' / 'section3_per_class_delta.png',
            dpi=150, bbox_inches='tight')
plt.show()

## 4 — Lock-in evaluation on the non-hidden test set

In [ ]:
# Evaluate the LOCKED system (best window) once on the untouched non-hidden test set
pred_nht_pp = utils.generate_predictions(our_nht_files, hgb, thresholds=thr,
                                          median_window=best_w, use_probs=True)
f1_nht_pp, per_nht_pp = utils.evaluate_split(pred_nht_pp, our_nht_files, ann_df_val)

# Reference: same system without post-processing
pred_nht_nopp = utils.generate_predictions(our_nht_files, hgb, thresholds=thr,
                                            median_window=1, use_probs=True)
f1_nht_nopp, per_nht_nopp = utils.evaluate_split(pred_nht_nopp, our_nht_files, ann_df_val)

print(f'Non-hidden test  HistGB + thresholds, no post-proc:  macro-F1 = {f1_nht_nopp:.4f}')
print(f'Non-hidden test  HistGB + thresholds, median w={best_w}:    macro-F1 = {f1_nht_pp:.4f}  '
      f'(delta = {f1_nht_pp - f1_nht_nopp:+.4f})')

summary = {
    'best_window': int(best_w),
    'window_sweep': sweep_df.to_dict('records'),
    'nht_macro_f1_no_postproc':    float(f1_nht_nopp),
    'nht_macro_f1_with_postproc':  float(f1_nht_pp),
    'delta_nht':                   float(f1_nht_pp - f1_nht_nopp),
}
with open(ART / 'section3_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print('saved -> artifacts/section3_summary.json')